# Listing photos and views: Steel-Dwass comparisons

**Kruskal-Wallis Test with Multiple Comparisons** · a worked example from the StatNex teaching guide

**The story.** A resale app looked at 21 listings for the same kind of jacket, posted with 1, 3, 5 or 10 photos, and counted how many people viewed each listing in its first week.

**The question.** Do listings with more photos get more views, and how many photos are enough?

**To run it**, choose **Runtime → Run all**. Colab first warns that this notebook was not written by Google; it reads none of your data, so choose **Run anyway**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `06_nonparametric/kruskal_wallis.csv` (6 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
1 photo,3 photos,5 photos,10 photos
88.0,118.0,135.0,148.0
102.0,131.0,150.0,170.0
95.0,109.0,128.0,139.0
120.0,140.0,162.0,165.0
76.0,,141.0,158.0
110.0,,155.0,

## Libraries

In [ ]:
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# One column per level. Text and blank cells are skipped, so levels may differ in size.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
levels = ["1 photo", "3 photos", "5 photos", "10 photos"]
groups = [pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float) for name in levels]
sizes = np.array([len(g) for g in groups])

## Summary by level

In [ ]:
# All values are ranked together (ties share the average rank); each level's mean rank
# is what the test compares.
ranks = stats.rankdata(np.concatenate(groups))
mean_ranks = [part.mean() for part in np.split(ranks, np.cumsum(sizes)[:-1])]
summary = pd.DataFrame({"n": sizes, "mean rank": mean_ranks}, index=levels).T
print(summary)

## Kruskal-Wallis test

In [ ]:
# SciPy's H carries the tie correction; it is compared with chi-square on k - 1 df.
h, p = stats.kruskal(*groups)
df = len(groups) - 1
print(f"\nchi-square = {h:.6f}, df = {df}, P = {p:.6g}")


# Wilcoxon's rank-sum statistic for one pair, ranked on its own (not with the other
# levels), standardised with the tie-corrected variance. Positive when the second
# group sits higher.
def rank_sum_z(a, b):
    combined = np.concatenate([a, b])
    total = len(combined)
    r = stats.rankdata(combined)
    expected = len(b) * (total + 1) / 2
    variance = len(a) * len(b) / (total * (total - 1)) * ((r**2).sum() - total * (total + 1) ** 2 / 4)
    return (r[len(a):].sum() - expected) / np.sqrt(variance)

## Multiple comparison: Steel-Dwass

In [ ]:
# Every pair, each ranked on its own; |z| x sqrt(2) is judged against the studentised
# range for k groups at infinite df (the rank statistic is asymptotically normal).
rows = []
for i, j in combinations(range(len(groups)), 2):
    z = rank_sum_z(groups[i], groups[j])
    rows.append([levels[i], levels[j], z,
                 stats.studentized_range.sf(abs(z) * np.sqrt(2), len(groups), np.inf)])
pairs = pd.DataFrame(rows, columns=["level 1", "level 2", "z", "P"])
print()
print(pairs.to_string(index=False))

## What the result says

The Kruskal-Wallis test compares groups by ranks, the positions of the listings when sorted by views. Its chi-square statistic is 14.89 and the P value is 0.002, so the photo counts differ somewhere. The P value is the chance of gaps this large if photos made no difference.

The Steel-Dwass method then compares every pair of photo counts. For each pair it ranks only the listings of those two groups, and it adjusts the P values because six pairs are tested. Two pairs are significant: 1 against 5 photos (P = 0.021) and 1 against 10 photos (P = 0.031). The others are not: 1 against 3 photos has P = 0.220, and 5 against 10 photos has P = 0.577.

Five or ten photos clearly beat one. Whether three beat one, or ten beat five, is not shown either way. Five photos look enough for now.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.